<a href="https://colab.research.google.com/github/darshanmgowda636337-collab/Insider_Threat_Detection/blob/main/Insider_Threat_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import gradio as gr

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("Environment initialized successfully.")

Environment initialized successfully.


In [ ]:
def generate_dataset(n=5000, seed=42):

    rng = np.random.default_rng(seed)

    users = [f"EMP-{i:03d}" for i in range(1, 101)]

    df = pd.DataFrame({
        "user_id": rng.choice(users, n),

        "login_hour": np.clip(
            rng.normal(9, 2, n).round(), 0, 23
        ).astype(int),

        "file_access_count": rng.poisson(12, n),

        "data_transfer_mb": rng.exponential(25, n),

        "failed_logins": rng.poisson(0.3, n),

        "usb_used": rng.choice(
            [0, 1], n, p=[0.92, 0.08]
        ),
    })

    # Create synthetic labels for demonstration.
    df["is_suspicious"] = 0

    suspicious_idx = rng.choice(
        df.index,
        size=int(n * 0.06),
        replace=False
    )

    df.loc[suspicious_idx, "login_hour"] = rng.choice(
        [0, 1, 2, 3, 23],
        len(suspicious_idx)
    )

    df.loc[suspicious_idx, "file_access_count"] = rng.integers(
        60, 150, len(suspicious_idx)
    )

    df.loc[suspicious_idx, "data_transfer_mb"] = rng.uniform(
        300, 1500, len(suspicious_idx)
    )

    df.loc[suspicious_idx, "failed_logins"] = rng.integers(
        3, 10, len(suspicious_idx)
    )

    df.loc[suspicious_idx, "usb_used"] = 1

    df.loc[suspicious_idx, "is_suspicious"] = 1

    return df


df = generate_dataset()

print("Dataset created successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset created successfully!
Dataset shape: (5000, 7)


,user_id,login_hour,file_access_count,data_transfer_mb,failed_logins,usb_used,is_suspicious
0,EMP-009,8,17,14.508947,0,0,0
1,EMP-078,8,7,25.364025,0,0,0
2,EMP-066,11,13,8.167730,1,0,0
3,EMP-044,11,10,19.158192,0,0,0
4,EMP-044,8,14,32.268759,0,0,0


In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nFirst five records:")
display(df.head())

print("\nNormal vs suspicious records:")
display(df["is_suspicious"].value_counts())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            5000 non-null   object 
 1   login_hour         5000 non-null   int64  
 2   file_access_count  5000 non-null   int64  
 3   data_transfer_mb   5000 non-null   float64
 4   failed_logins      5000 non-null   int64  
 5   usb_used           5000 non-null   int64  
 6   is_suspicious      5000 non-null   int64  
dtypes: float64(1), int64(5), object(1)
memory usage: 273.6+ KB

Missing values:


,Missing Values
user_id,0
login_hour,0
file_access_count,0
data_transfer_mb,0
failed_logins,0
usb_used,0
is_suspicious,0



First five records:


,user_id,login_hour,file_access_count,data_transfer_mb,failed_logins,usb_used,is_suspicious
0,EMP-009,8,17,14.508947,0,0,0
1,EMP-078,8,7,25.364025,0,0,0
2,EMP-066,11,13,8.167730,1,0,0
3,EMP-044,11,10,19.158192,0,0,0
4,EMP-044,8,14,32.268759,0,0,0



Normal vs suspicious records:


,count
is_suspicious,
0,4700
1,300


In [ ]:
FEATURES = [
    "login_hour",
    "file_access_count",
    "data_transfer_mb",
    "failed_logins",
    "usb_used"
]

X = df[FEATURES]

model = IsolationForest(
    n_estimators=200,
    contamination=0.06,
    random_state=42
)

model.fit(X)

df["anomaly_prediction"] = (
    model.predict(X) == -1
).astype(int)

df["anomaly_score"] = -model.score_samples(X)

print("Model training completed!")
print(
    "Detected anomalies:",
    df["anomaly_prediction"].sum()
)

display(
    df[
        FEATURES + [
            "is_suspicious",
            "anomaly_prediction",
            "anomaly_score"
        ]
    ].head(10)
)

Model training completed!
Detected anomalies: 300


,login_hour,file_access_count,data_transfer_mb,failed_logins,usb_used,is_suspicious,anomaly_prediction,anomaly_score
0,8,17,14.508947,0,0,0,0,0.385602
1,8,7,25.364025,0,0,0,0,0.389603
2,11,13,8.167730,1,0,0,0,0.403169
3,11,10,19.158192,0,0,0,0,0.370184
4,8,14,32.268759,0,0,0,0,0.369092
5,8,5,20.714921,0,0,0,0,0.422408
6,10,10,20.300033,0,0,0,0,0.360603
7,7,16,11.658786,1,0,0,0,0.420605
8,8,12,8.212364,0,0,0,0,0.358932
9,9,9,8.495424,0,0,0,0,0.364093


In [ ]:
# STEP 5: MODEL EVALUATION

y_true = df["is_suspicious"]
y_pred = df["anomaly_prediction"]

precision = precision_score(
    y_true, y_pred, zero_division=0
)

recall = recall_score(
    y_true, y_pred, zero_division=0
)

f1 = f1_score(
    y_true, y_pred, zero_division=0
)

print("========== MODEL EVALUATION ==========")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Normal", "Suspicious"],
        zero_division=0
    )
)

========== MODEL EVALUATION ==========
Precision : 1.0000
Recall    : 1.0000
F1 Score  : 1.0000

Confusion Matrix:
[[4700    0]
 [   0  300]]

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      4700
  Suspicious       1.00      1.00      1.00       300

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000



In [ ]:
# STEP 6: RISK SCORING

df["risk_score"] = (
    20 * df["login_hour"].isin(
        [0, 1, 2, 3, 23]
    ).astype(int)

    + 20 * (df["file_access_count"] > 40).astype(int)

    + 25 * (df["data_transfer_mb"] > 200).astype(int)

    + 15 * (df["failed_logins"] >= 3).astype(int)

    + 20 * df["usb_used"]
).clip(0, 100)

# Flag anomalies with a risk score of at least 40.
df["alert"] = (
    (df["anomaly_prediction"] == 1)
    & (df["risk_score"] >= 40)
).astype(int)

alerts = df[df["alert"] == 1].sort_values(
    "risk_score",
    ascending=False
)

print("Total activity records:", len(df))
print("Detected anomalies:", df["anomaly_prediction"].sum())
print("High-priority alerts:", len(alerts))

display(
    alerts[
        [
            "user_id",
            "login_hour",
            "file_access_count",
            "data_transfer_mb",
            "failed_logins",
            "usb_used",
            "risk_score"
        ]
    ].head(15)
)

Total activity records: 5000
Detected anomalies: 300
High-priority alerts: 300


,user_id,login_hour,file_access_count,data_transfer_mb,failed_logins,usb_used,risk_score
4993,EMP-056,3,78,1424.961930,3,1,100
29,EMP-045,0,61,741.766733,7,1,100
36,EMP-086,1,120,1204.666960,8,1,100
46,EMP-045,23,71,1258.138612,6,1,100
73,EMP-013,0,81,474.752327,3,1,100
109,EMP-071,1,90,949.838515,9,1,100
113,EMP-046,0,143,1498.188035,7,1,100
126,EMP-008,2,78,465.948415,6,1,100
133,EMP-056,2,122,1268.916092,4,1,100
136,EMP-061,3,95,979.227824,9,1,100


In [ ]:
# STEP 7: BEHAVIOURAL ANOMALY VISUALIZATION

fig = px.scatter(
    df,
    x="file_access_count",
    y="data_transfer_mb",
    color="anomaly_prediction",
    hover_data=[
        "user_id",
        "login_hour",
        "failed_logins",
        "usb_used",
        "risk_score"
    ],
    title="Insider Threat Detection: Behavioural Anomalies",
    labels={
        "file_access_count": "File Access Count",
        "data_transfer_mb": "Data Transfer (MB)",
        "anomaly_prediction": "Anomaly Detected"
    }
)

fig.show()

In [ ]:
# STEP 8: INTERACTIVE INSIDER THREAT DASHBOARD

def investigate_activity(
    login_hour,
    file_access_count,
    data_transfer_mb,
    failed_logins,
    usb_used
):

    activity = pd.DataFrame([{
        "login_hour": int(login_hour),
        "file_access_count": int(file_access_count),
        "data_transfer_mb": float(data_transfer_mb),
        "failed_logins": int(failed_logins),
        "usb_used": int(usb_used)
    }])

    prediction = int(
        model.predict(activity)[0] == -1
    )

    anomaly_score = float(
        -model.score_samples(activity)[0]
    )

    risk_score = int(
        20 * (int(login_hour) in [0, 1, 2, 3, 23])
        + 20 * (int(file_access_count) > 40)
        + 25 * (float(data_transfer_mb) > 200)
        + 15 * (int(failed_logins) >= 3)
        + 20 * int(usb_used)
    )

    return {
        "User Behaviour": (
            "Anomaly detected"
            if prediction
            else "No anomaly detected"
        ),
        "Anomaly Score": round(anomaly_score, 4),
        "Risk Score": risk_score,
        "Recommended Action": (
            "Review this activity manually"
            if prediction and risk_score >= 40
            else "No high-priority alert from current rules"
        )
    }


demo = gr.Interface(
    fn=investigate_activity,

    inputs=[
        gr.Slider(
            0, 23, value=9, step=1,
            label="Login Hour"
        ),

        gr.Slider(
            0, 200, value=12, step=1,
            label="File Access Count"
        ),

        gr.Number(
            value=25,
            label="Data Transfer (MB)"
        ),

        gr.Slider(
            0, 15, value=0, step=1,
            label="Failed Login Attempts"
        ),

        gr.Radio(
            choices=[0, 1],
            value=0,
            label="USB Used? (0 = No, 1 = Yes)"
        )
    ],

    outputs=gr.JSON(label="Analysis Results"),

    title="Insider Threat Detection System",

    description=(
        "ML-powered user behaviour analysis using "
        "Isolation Forest and rule-based risk scoring. "
        "All alerts require human review."
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e1ce054cae71103e68.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
